In [ ]:
import os
import re

import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
STAT_PATH = "../experiments"
CLIENTS_DYNAMIC_NAME = os.path.join("exp", "clients_dynamic")
PARAMS_NAME = "parameters.xml"
EXP_NAME = "run_"


In [ ]:
def read_total_clients(params_path):
    with open(params_path, "r", encoding="utf-8") as file:
        content = file.read()

    match = re.search(r"<n_clients>\s*([0-9]+)\s*</n_clients>", content)
    if match:
        return int(match.group(1))

    return None


def read_active_clients(file_path):
    df = pd.read_csv(file_path, header=None, names=["delta"])
    df["active_clients"] = df["delta"].cumsum()
    df["time_hours"] = df.index / 3600.0
    return df


def plot_active_clients(ax, df, label):
    ax.plot(df["time_hours"], df["active_clients"], label=label)
    ax.set_title("Число хостов в онлайне")
    ax.set_xlabel("Время, ч")
    ax.set_ylabel("Активные хосты, ед")
    ax.grid(True, which="both", linestyle="--", linewidth=0.5)


In [ ]:
folders = sorted(
    folder
    for folder in os.listdir(STAT_PATH)
    if os.path.isdir(os.path.join(STAT_PATH, folder)) and folder.startswith(EXP_NAME)
)

if not folders:
    print(f"there is no folders in {STAT_PATH}")
else:
    print(f"found {folders}")

datasets = []

for folder in folders:
    full_path = os.path.join(STAT_PATH, folder)
    clients_dynamic_path = os.path.join(full_path, CLIENTS_DYNAMIC_NAME)

    if not os.path.exists(clients_dynamic_path):
        print(f"skip {folder}: {CLIENTS_DYNAMIC_NAME} not found, rerun simulation with updated boinc_simulator.c")
        continue

    total_clients = None
    params_path = os.path.join(full_path, PARAMS_NAME)
    if os.path.exists(params_path):
        total_clients = read_total_clients(params_path)

    label = folder
    if total_clients is not None:
        label = f"{folder} ({total_clients} clients)"

    datasets.append((label, read_active_clients(clients_dynamic_path)))

if not datasets:
    print("there is no active hosts data to plot")
else:
    fig, ax = plt.subplots(figsize=(12, 8))

    for label, df in datasets:
        plot_active_clients(ax, df, label)

    plt.tight_layout()
    plt.show()
